In [12]:
import os
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    PegasusTokenizer,
    PegasusForConditionalGeneration,
    Trainer,
    TrainingArguments
)

In [13]:
articles_path = r"C:\Users\EileneAnnaKuriakose\Desktop\BBC News Summary\BBC News Summary\News Articles"
summaries_path = r"C:\Users\EileneAnnaKuriakose\Desktop\BBC News Summary\BBC News Summary\Summaries"

articles = []
summaries = []

categories = ["business", "entertainment", "politics", "sport", "tech"]

for category in categories:
    article_folder = os.path.join(articles_path, category)
    summary_folder = os.path.join(summaries_path, category)

    for filename in os.listdir(article_folder):

        article_file = os.path.join(article_folder, filename)
        summary_file = os.path.join(summary_folder, filename)

        with open(article_file, "r", encoding="latin-1") as f:
            article = f.read()

        with open(summary_file, "r", encoding="latin-1") as f:
            summary = f.read()

        articles.append(article)
        summaries.append(summary)

In [14]:
df = pd.DataFrame({
    "article": articles,
    "summary": summaries
})

print(df.shape)
df.head()

(2225, 2)


,article,summary
0,Ad sales boost Time Warner profit\n\nQuarterly...,TimeWarner said fourth quarter sales rose 2% t...
1,Dollar gains on Greenspan speech\n\nThe dollar...,The dollar has hit its highest level against t...
2,Yukos unit buyer faces loan claim\n\nThe owner...,Yukos' owner Menatep Group says it will ask Ro...
3,High fuel prices hit BA's profits\n\nBritish A...,"Rod Eddington, BA's chief executive, said the ..."
4,Pernod takeover talk lifts Domecq\n\nShares in...,Pernod has reduced the debt it took on to fund...


In [15]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [16]:
train_dataset = train_dataset.select(range(100))
test_dataset = test_dataset.select(range(20))

In [17]:
tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-xsum")

In [18]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["article"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [19]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [20]:
from transformers import PegasusForConditionalGeneration

model = PegasusForConditionalGeneration.from_pretrained(
    "google/pegasus-xsum"
)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
training_args = TrainingArguments(
    output_dir="./pegasus_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

In [24]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
C:\Users\EileneAnnaKuriakose\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.893999,1.114531


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=25, training_loss=2.129797019958496, metrics={'train_runtime': 887.4502, 'train_samples_per_second': 0.113, 'train_steps_per_second': 0.028, 'total_flos': 144473220710400.0, 'train_loss': 2.129797019958496, 'epoch': 1.0})

In [25]:
article = df["article"][0]

inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("ARTICLE:\n")
print(article[:1000])

print("\n\nGENERATED SUMMARY:\n")
print(summary)

ARTICLE:

Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (Â£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to 

In [28]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

loaded_tokenizer = PegasusTokenizer.from_pretrained("./saved_model")
loaded_model = PegasusForConditionalGeneration.from_pretrained("./saved_model")

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

In [29]:
article = df["article"][1]

inputs = loaded_tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = loaded_model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

All images are copyrighted.
